# FIFO Buffer Concept

1. Need to understand for FIFO buffer, mutex in python
2. Could abuse the GIL removal for coursework?
3. Could do it in C/C++?
4. Get C Kernel working in Notebook.
5. Add unit tests?

## FIFO Buffer Explanation for Part 1

A FIFO buffer is a data structure that stores elements in the order they are received, such that the first element written into the buffer is the first element to be read out.

FIFO buffer serves as temporary storage between two processes that may operate at different speeds. If a peripheral device produces data faster than the processor can consume it, the buffer holds the incoming data until the processor is ready to retrieve it. Without such a buffer, data would be lost.

The buffer is typically implemented as a circular array in memory. Two indices are maintained: one indicating the next position to be written to, and another indicating the next position to be read from. When data is added, it is placed at the write index, which then advances forward. When data is retrieved, it is taken from the read index, which also advances. Upon reaching the end of the array, each index wraps back to the beginning, allowing the memory to be reused continuously.

A dynamically set buffer means that the number of memory slots is not fixed at compile time. The size is determined at runtime, allowing the system to allocate only as much memory as is needed for a given application.

Additionally, the buffer must track whether it is full or empty to prevent data from being overwritten before it is read, or from being read when no valid data is present.

In [ ]:
# Potential Imports??
# FIFO Buffer class
import sys
import os
import numpy as np

In [1]:
# FIFO Buffer Class Creation
class FIFO:
    def __init__(self, size):          # Create a buffer with a given number of slots
        self.data = [None] * size      # Create the array with empty slots
        self.head = 0                  # Where to write next
        self.tail = 0                  # Where to read next
        self.count = 0                 # How many items are stored
        self.size = size               # Total number of slots

    # Push Operation Method
    def push(self, value):             # Add a value to the buffer
        if self.count == self.size:    # Check if full (exception,  but I need to add tests as well)
            print(f"Full buffer. Cannot push {value}.")
            return False
        self.data[self.head] = value   # Write at head pointer position
        self.head = (self.head + 1) % self.size  # Move head pointer forward, wrap around if needed
        self.count += 1                # One more item stored
        return True

    # Pop Operation Method
    def pop(self):                     # Remove a value from the buffer
        if self.count == 0:            # Check if empty exception
            print("Empty Buffer. Cannot pop.")
            return None
        value = self.data[self.tail]   # Read from tail position
        self.tail = (self.tail + 1) % self.size  # Move tail pointer forward, wrap if needed
        self.count -= 1                # One less item stored
        return value

## Code tests

In [3]:
import unittest  # Import the testing framework (don't know if these imports are allowed)

class TestFIFO(unittest.TestCase):

    # Test 1: Normal push and pop in order
    def test_push_pop_order(self):
        fifo = FIFO(3)
        fifo.push(10)
        fifo.push(20)
        fifo.push(30)
        self.assertEqual(fifo.pop(), 10)  # First in, first out
        self.assertEqual(fifo.pop(), 20)
        self.assertEqual(fifo.pop(), 30)

    # Test 2: Overflow - push when full
    def test_overflow(self):
        fifo = FIFO(3)
        fifo.push(1)
        fifo.push(2)
        fifo.push(3)
        result = fifo.push(4)             # Should fail
        self.assertFalse(result)          # Push returns False
        self.assertEqual(fifo.count, 3)   # Count unchanged
        self.assertEqual(fifo.pop(), 1)   # Original data intact

    # Test 3: Underflow - pop when empty
    def test_underflow(self):
        fifo = FIFO(3)
        result = fifo.pop()               # Nothing to pop
        self.assertIsNone(result)         # Returns None
        self.assertEqual(fifo.count, 0)   # Count unchanged

    # Test 4: Wraparound
    def test_wraparound(self):
        fifo = FIFO(3)
        fifo.push(1)
        fifo.push(2)
        fifo.push(3)
        fifo.pop()                        # Free up a slot
        fifo.pop()                        # Free up another slot
        fifo.push(40)                     # Should wrap around
        fifo.push(50)                     # Should wrap around
        self.assertEqual(fifo.pop(), 3)   # Remaining from first fill
        self.assertEqual(fifo.pop(), 40)  # Wrapped value
        self.assertEqual(fifo.pop(), 50)  # Wrapped value

# Run tests in Jupyter notebook
unittest.main(argv=[''], exit=False, verbosity=2)

test_overflow (__main__.TestFIFO.test_overflow) ... ok
test_push_pop_order (__main__.TestFIFO.test_push_pop_order) ... ok
test_underflow (__main__.TestFIFO.test_underflow) ... ok
test_wraparound (__main__.TestFIFO.test_wraparound) ... ok

----------------------------------------------------------------------
Ran 4 tests in 0.009s

OK


Full buffer. Cannot push 4.
Empty Buffer. Cannot pop.
